# 03 Checkpoint verifier restore smoke

Independently verify and restore an exact private checkpoint candidate.

This private `orchestrator_protected` notebook is generated from a reviewed Python template. It receives exact input versions and secrets through Kaggle runtime inputs; no credential is embedded in this notebook or written to its output.

In [ ]:
from __future__ import annotations

import hashlib
import os
import subprocess
import sys
from pathlib import Path

EXPECTED_SOURCE_SHA256 = '2e29576c200979b982abf4d19f0a7aeb6523fdbf40a0dd019a1c59192b9cf82e'
RUNTIME_CONTRACT = 'my-data-hub-checkpoint-restore-smoke.v1'
wheel = Path(os.environ.get('MY_DATA_HUB_WHEEL_PATH', ''))
if not wheel.is_file() or wheel.suffix != '.whl':
    raise RuntimeError('exact private my-data-hub wheel input is required')
expected_wheel_sha = os.environ.get('MY_DATA_HUB_WHEEL_SHA256', '')
if (len(expected_wheel_sha) != 64 or 
        hashlib.sha256(wheel.read_bytes()).hexdigest() != expected_wheel_sha):
    raise RuntimeError('my-data-hub wheel hash mismatch')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-deps', '--disable-pip-version-check', str(wheel)],
    check=True,
)

In [ ]:
PRIMARY_SOURCE = '"""Primary source for an independent checkpoint restore-smoke notebook."""\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nimport psycopg\n\nfrom my_data_hub.checkpoints.manifest import load_and_verify\nfrom my_data_hub.checkpoints.publisher import assert_restore_equality\nfrom my_data_hub.checkpoints.restore_probe import collect_restore_probe\nfrom my_data_hub.hashing import canonical_json_bytes, sha256_file\n\n\ndef _path(name: str) -> Path:\n    path = Path(os.environ.get(name, ""))\n    if not path.exists() or path.is_symlink():\n        raise RuntimeError(f"required exact artifact is absent: {name}")\n    return path\n\n\ndef main() -> int:\n    package = _path("MY_DATA_HUB_CHECKPOINT_DIRECTORY")\n    manifest_path = _path("MY_DATA_HUB_CHECKPOINT_MANIFEST")\n    manifest = load_and_verify(manifest_path, package)\n    database_url = os.environ.get("MY_DATA_HUB_RESTORE_DATABASE_URL", "")\n    if not database_url.startswith(("postgresql://", "postgres://")):\n        raise RuntimeError("isolated restore database URL is required")\n    with psycopg.connect(database_url, connect_timeout=15) as connection:\n        observed = collect_restore_probe(connection, tuple(sorted(manifest.restore_probe.row_counts)))\n    assert_restore_equality(manifest, observed)\n    receipt = {\n        "schema_version": "my-data-hub-checkpoint-restore-smoke.v1",\n        "checkpoint_id": str(manifest.checkpoint_id),\n        "manifest_sha256": manifest.manifest_sha256,\n        "manifest_file_sha256": sha256_file(manifest_path),\n        "ok": True,\n        "observed": observed,\n    }\n    output = Path("/kaggle/working/checkpoint-restore-receipt.json")\n    output.write_bytes(canonical_json_bytes(receipt))\n    return 0\n'
if hashlib.sha256(PRIMARY_SOURCE.encode()).hexdigest() != EXPECTED_SOURCE_SHA256:
    raise RuntimeError('embedded primary source hash mismatch')
exec(compile(PRIMARY_SOURCE, '<my-data-hub-primary-source>', 'exec'), globals())

In [ ]:
raise SystemExit(globals()['main']())